# 12: Forward log

The running out-of-sample record, extended once a month. To update it, set
END to the last complete UTC day, run the notebook and commit it, so each
month's numbers are on record before the next month's data exists. Commit
dates are set by the machine that makes the commit, so for anyone else they
only count once each month's commit is pushed somewhere public. If the
archive has not published a day yet, the notebook stops before logging
anything; run it again a day later.

Nothing here is tuned. Three records are kept:

- The frozen book (v1), from 2026-07-07, the first day the research data
  did not have: its coin list, sleeves, walk-forward weights and costs.
  Notebook 08 ran its first 82 days.
- v2 (notebook 10), from 2026-10-01, the first full day after the
  repository, with v2's rules, was first pushed to GitHub (the start was
  2026-09-28 until then). A Carry position held into a day notebook 09's
  20% rule drops earns what its contract did: its own move and funding, or
  nothing once it has stopped trading. That measurement, and the filling of
  two holes in the archive's 2022 files, came after v2's first run and
  before its test began (notebook 11).
- v2's two sleeves on their own, since notebook 10 found that its
  Orderflow sleeve fails the gate on the every-pair universe.

History up to 2026-09-26 comes from notebook 09. Later days come from the
Binance archive the same way, so a coin delisted during a month stays in
the record until its last trade. Funding for the current month, which the
archive has not written yet, comes from the API.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

import sys
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from quantlib import data, fetch, metrics, robustness, strategies

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
ARCHIVE, ALL = RAW / "archive", PROC / "all_pairs"

END = "2026-09-26"                 # set to the last complete UTC day before each run
BASE_END = "2026-09-26"            # notebook 09's data ends here
V1_START, V2_START = "2026-07-07", strategies.V2_START
assert pd.Timestamp(END) < pd.Timestamp.now(tz="UTC").tz_localize(None).normalize(), "END must be a finished day"

## Data

Notebook 09's panels, plus any days after 2026-09-26. The new data is
fetched from 2026-09-01, so a pair listed in September, which had no monthly
file yet when notebook 09 listed the archive, keeps its first days; where
it overlaps notebook 09's data, it has to match.

In [2]:
def panels_from(name_dir: Path, names: list[str]) -> dict:
    return {n: pd.read_parquet(name_dir / f"{n}.parquet") for n in names}


SPOT = ["price", "returns", "dollar_volume", "taker_imbalance", "high", "low"]
PERP = ["perp_price", "perp_high", "perp_low", "perp_returns", "perp_dollar_volume", "funding",
        "perp_returns_traded", "funding_traded"]
allp = panels_from(ALL, SPOT + PERP)
rl = panels_from(ALL / "research_list", ["price", "returns", "dollar_volume", "taker_imbalance", "funding"])

if pd.Timestamp(END) > pd.Timestamp(BASE_END):
    start = "2026-09-01"      # the month notebook 09's data ends in
    spot_pairs = fetch.archive_pairs("spot")
    keep = set(data.tradable_bases([p[:-4] for p in spot_pairs]))
    research_coins = list(rl["price"].columns)
    spot_keep = [p for p in spot_pairs if p[:-4] in keep or p[:-4] in research_coins]
    new_spot = fetch.refresh(RAW / f"log_spot_{END}.pkl.zip",
                             lambda: fetch.fetch_archive("spot", spot_keep, start, END, ARCHIVE, workers=24),
                             max_age_days=float("inf"))
    perp_names = data.perp_to_spot([p[:-4] for p in fetch.archive_pairs("um")], sorted(keep))
    new_um = fetch.refresh(RAW / f"log_um_{END}.pkl.zip",
                           lambda: fetch.fetch_archive("um", sorted(p + "USDT" for p in perp_names), start, END,
                                                       ARCHIVE, workers=24),
                           max_age_days=float("inf"))

    # The archive sometimes publishes a day late (the 2026-09-26 perp bars did).
    # Stop, and drop this run's caches so the next run downloads again, if any
    # new day is short of bars; otherwise a late day would be logged half-empty
    # and quietly revised by the next month's run.
    short = {}
    for name, raw in [("spot", new_spot), ("perp", new_um)]:
        s = fetch.short_days(raw, start, END)
        s = s[s.index > pd.Timestamp(BASE_END)]
        if len(s):
            short[name] = [d.date() for d in s.index]
    if short:
        for f in RAW.glob(f"log_*_{END}.pkl.zip"):
            f.unlink()
        raise RuntimeError(f"the archive is short of bars on {short}; run again once it has published them")
    fund_names = sorted(set(perp_names) | set(rl["funding"].columns))

    def archive_funding() -> pd.DataFrame:
        try:
            return fetch.fetch_archive("funding", [p + "USDT" for p in fund_names], start, END, ARCHIVE, workers=24)
        except RuntimeError:          # no monthly funding file covers these days yet
            return pd.DataFrame()

    fund_arch = fetch.refresh(RAW / f"log_funding_{END}.pkl.zip", archive_funding, max_age_days=float("inf"))
    api_start = (fund_arch.index.max() + pd.Timedelta(days=1)).strftime("%Y-%m-%d") if len(fund_arch) else start
    fund_api = fetch.refresh(RAW / f"log_funding_api_{END}.pkl.zip",
                             lambda: fetch.fetch_funding(fund_names, start=api_start,
                                                         end=(pd.Timestamp(END) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")),
                             max_age_days=float("inf")).loc[api_start:END]
    new_funding = pd.concat([fund_arch, fund_api]).sort_index()

    new_all = data.archive_panels(new_spot.loc[:, new_spot.columns.get_level_values(0).isin(keep)],
                                  new_um, new_funding, perp_names)
    new_rl = data.to_panels(new_spot.loc[:, new_spot.columns.get_level_values(0).isin(research_coins)])
    new_rl["funding"] = new_funding.reindex(columns=rl["funding"].columns)
    overlap = slice(start, BASE_END)
    for name, old, new in [("price", allp["price"], new_all["price"]), ("funding", allp["funding"], new_all["funding"])]:
        a, b = old.loc[overlap], new.reindex(columns=old.columns).loc[overlap]
        both = a.notna() & b.notna()
        assert np.allclose(a[both].fillna(0), b[both].fillna(0), rtol=1e-9, atol=1e-12), f"{name} overlap differs"

    def join(old: dict, new: dict) -> dict:
        """Old days, then new ones; the overlap also fills days the archive published late."""
        after = pd.Timestamp(BASE_END) + pd.Timedelta(days=1)
        return {k: pd.concat([old[k], new[k].loc[after:]]).sort_index().combine_first(new[k])
                for k in old if k in new}

    allp, rl = join(allp, new_all), join(rl, new_rl)
    first = {p[:-4]: min(fetch.archive_months("spot", p)) for p in spot_pairs if p[:-4].endswith("B")}
    unknown = sorted(b for b, m in first.items() if m >= pd.Period("2026-06", "M") and b not in data.TOKENIZED_STOCKS)
    print("B-suffixed listings not in data.TOKENIZED_STOCKS (check whether they are stocks):", unknown)

print("data ends", allp["price"].index.max().date())

data ends 2026-09-26


## The frozen book (v1)

The research coin list and its 55 perps, the three frozen sleeves and
notebook 06's combination rules, on archive data.

In [3]:
uni_v1 = data.build_universe(rl, top_n=100, min_adv_usd=1_000_000)
v1 = pd.DataFrame({
    "seasonality": strategies.seasonal_momentum_sleeve(rl["price"], rl["returns"], uni_v1),
    "orderflow": strategies.orderflow_sleeve(rl["taker_imbalance"], rl["returns"], uni_v1),
    "carry": strategies.carry_sleeve(rl["funding"].reindex(columns=uni_v1.columns), rl["returns"], uni_v1),
}).dropna()
v1_wf, _ = robustness.walk_forward_weights(v1, train_days=756, step_days=63, min_train=252)
v1_ew = v1.mean(axis=1)

# notebook 08 ran the first 82 days on the API data; the archive rebuild should agree
nb08 = pd.read_parquet(PROC / "forward_book.parquet").loc[V1_START:"2026-09-26"]
agree = pd.concat([v1_wf.loc[V1_START:"2026-09-26"], nb08["walk_forward"]], axis=1).dropna()
print(f"v1 walk-forward vs notebook 08 over its 82 days: largest daily gap {agree.diff(axis=1).iloc[:, 1].abs().max():.2e}, "
      f"total return {(1 + agree.iloc[:, 0]).prod() - 1:+.4f} against {(1 + agree.iloc[:, 1]).prod() - 1:+.4f}")

v1 walk-forward vs notebook 08 over its 82 days: largest daily gap 1.39e-17, total return -0.0594 against -0.0594


## v2

In [4]:
uni_v2 = data.build_universe(allp, top_n=100, min_adv_usd=1_000_000)
v2 = strategies.v2_sleeves(allp["taker_imbalance"], allp["returns"],
                           allp["perp_returns_traded"].reindex(columns=uni_v2.columns),
                           allp["funding"].reindex(columns=uni_v2.columns), uni_v2,
                           pnl_funding=allp["funding_traded"].reindex(columns=uni_v2.columns))
v2_book = strategies.v2_book(v2)
# the months before V2_START are only history for v2's first positions: its
# P&L on them is dropped here, before anything is recorded or shown
v2, v2_book = v2.loc[V2_START:], v2_book.loc[V2_START:]
assert v2.empty or v2.index.min() >= pd.Timestamp(V2_START)
btc = allp["returns"]["BTC"]

## The record

In [5]:
records = {"v1, walk-forward": (v1_wf, V1_START), "v1, equal weight": (v1_ew, V1_START),
           "v2 book": (v2_book, V2_START), "v2 orderflow": (v2["orderflow"], V2_START),
           "v2 carry": (v2["carry"], V2_START)}


def record(r: pd.Series, start: str) -> dict:
    r = r.loc[start:END].dropna()
    if len(r) < 2:
        return {"from": start, "days": len(r)}
    b = btc.reindex(r.index)
    return {"from": start, "days": len(r), "total_return": (1 + r).prod() - 1, "ann_vol": metrics.ann_vol(r),
            "sharpe": metrics.sharpe(r), "t_stat": metrics.sharpe(r) * np.sqrt(len(r) / 365),
            "max_dd": metrics.max_drawdown(r), "beta_btc": np.cov(r, b)[0, 1] / b.var(),
            "btc_total_return": (1 + b).prod() - 1}


log = pd.DataFrame({k: record(s, start) for k, (s, start) in records.items()}).T
log

,from,days,total_return,ann_vol,sharpe,t_stat,max_dd,beta_btc,btc_total_return
"v1, walk-forward",2026-07-07,82,-0.059445,0.243073,-1.006607,-0.477112,-0.093816,-0.120219,0.318402
"v1, equal weight",2026-07-07,82,0.051348,0.594219,0.634281,0.300637,-0.113406,-0.20028,0.318402
v2 book,2026-10-01,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
v2 orderflow,2026-10-01,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
v2 carry,2026-10-01,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Month by month (total return in each calendar month):

In [6]:
daily = pd.DataFrame({k: s.loc[start:END] for k, (s, start) in records.items()} | {"BTC": btc.loc[V1_START:END]})
monthly = (1 + daily).resample("ME").prod(min_count=1) - 1
monthly.index = monthly.index.strftime("%Y-%m")
monthly.round(4)

,"v1, walk-forward","v1, equal weight",v2 book,v2 orderflow,v2 carry,BTC
Date,,,,,,
2026-07,0.0378,0.1310,NaN,NaN,NaN,-0.0180
2026-08,-0.0822,-0.0673,NaN,NaN,NaN,0.2495
2026-09,-0.0125,-0.0034,NaN,NaN,NaN,0.0745


Save the daily record (not committed; the executed notebook is the record).
Days already in the last saved record should not change; any that did, say
because the API lacked a day's funding when the month was first logged, are
listed first.

In [7]:
last_path = PROC / "forward_log.parquet"
if last_path.exists():
    last = pd.read_parquet(last_path)
    cols, days = last.columns.intersection(daily.columns), last.index.intersection(daily.index)
    changed = ~np.isclose(daily.loc[days, cols].to_numpy(), last.loc[days, cols].to_numpy(),
                          rtol=0, atol=1e-12, equal_nan=True)
    revised = [d.date() for d in days[changed.any(axis=1)]]
    print(f"days of the last saved record that changed: {len(revised)}", revised[:10])
daily.to_parquet(PROC / "forward_log.parquet")
print("saved forward_log.parquet through", END)

days of the last saved record that changed: 0 []
saved forward_log.parquet through 2026-09-26
